In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
import sys
!{sys.executable} -m pip install sentence-transformers

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.decomposition import PCA

from sentence_transformers import SentenceTransformer, util
import sys
import os
os.chdir("..")
print(os.getcwd())
sys.path.append(os.path.abspath(".."))

print(" All libraries imported successfully.")


c:\Users\Administrator\Desktop\crop project\croprecommendationproject
 All libraries imported successfully.


In [2]:
from src.preprocessing import CropPreprocessor

preprocessor = CropPreprocessor()

data = preprocessor.preprocess("data/raw/Crop_recommendation.csv")

Cleaned dataset saved to: data/processed/cleaned_crop_data.csv


In [3]:
# Downloads ~80MB on first run, cached afterwards
bert_model = SentenceTransformer('all-MiniLM-L6-v2')
print(" BERT model loaded:", bert_model.get_sentence_embedding_dimension(), "dimensions")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

 BERT model loaded: 384 dimensions


In [4]:

# Values used in generate_feature_vector() below are calibrated to dataset means.

intent_templates = {

    
    'soil_low_nitrogen': [
        "soil with very low nitrogen content",
        "nitrogen deficient ground poor in N",
        "sandy or light soil lacking nutrients",
        "infertile soil low in nitrogen",
    ],
    'soil_medium_nitrogen': [
        "soil with moderate nitrogen level",
        "average nitrogen content in the field",
        "balanced soil nutrients medium N",
    ],
    'soil_high_nitrogen': [
        "soil rich in nitrogen very fertile",
        "high nitrogen content dark rich earth",
        "nitrogen-rich fertile loamy field",
    ],

    
    'temp_very_cold': [
        "very cold weather below 15 degrees",
        "freezing temperatures frost conditions",
        "extremely cold climate icy",
    ],
    'temp_cool': [
        "cool mild weather around 18 to 22 degrees",
        "moderate cold temperate conditions",
        "cool climate spring-like weather",
    ],
    'temp_warm': [
        "warm weather between 23 and 28 degrees",
        "pleasant warm conditions subtropical",
        "mild warm tropical weather",
    ],
    'temp_hot': [
        "hot weather above 30 degrees",
        "very hot scorching heat summer",
        "high temperature arid hot climate",
    ],
    'temp_very_hot': [
        "extremely hot above 35 degrees",
        "scorching extreme heat desert conditions",
        "very high temperature dry heat",
    ],

    
    'humidity_very_low': [
        "very dry air extremely low humidity",
        "arid dry conditions almost no moisture",
        "desert-like very low humidity",
    ],
    'humidity_low': [
        "low humidity dry air conditions",
        "semi-arid dry moderate humidity",
        "relatively dry air low moisture",
    ],
    'humidity_moderate': [
        "moderate humidity balanced moisture",
        "average humidity comfortable conditions",
        "normal moisture levels in air",
    ],
    'humidity_high': [
        "high humidity moist humid conditions",
        "very humid tropical moist air",
        "muggy damp high moisture environment",
    ],
    'humidity_very_high': [
        "extremely high humidity almost saturated",
        "very moist tropical rainforest conditions",
        "near 100 percent humidity waterlogged air",
    ],

    
    'rainfall_very_low': [
        "very little rain almost no rainfall",
        "drought conditions extremely scarce precipitation",
        "barely any rain very dry season",
    ],
    'rainfall_low': [
        "low rainfall below average precipitation",
        "not much rain semi-arid conditions",
        "sparse rainfall dry season",
    ],
    'rainfall_moderate': [
        "moderate rainfall average precipitation",
        "normal rain levels balanced moisture",
        "regular seasonal rainfall",
    ],
    'rainfall_high': [
        "high rainfall heavy rains",
        "lots of rain heavy precipitation",
        "rainy season heavy showers frequent rain",
    ],
    'rainfall_very_high': [
        "extremely heavy rainfall flooding",
        "monsoon heavy downpours very high precipitation",
        "near 300mm rainfall tropical monsoon",
    ],

   
    'ph_acidic': [
        "acidic soil low pH below 6",
        "highly acidic ground sour soil",
        "acid soil pH around 5 or less",
    ],
    'ph_slightly_acidic': [
        "slightly acidic soil pH around 6",
        "mildly acidic soil common farming land",
        "near neutral slightly acidic ground",
    ],
    'ph_neutral': [
        "neutral soil pH around 6.5 to 7",
        "balanced neutral soil neither acidic nor alkaline",
        "ideal pH neutral fertile soil",
    ],
    'ph_alkaline': [
        "alkaline soil high pH above 7",
        "basic alkaline ground high pH soil",
        "soil pH above 7.5 alkaline conditions",
    ],
}


encoded_templates = {}
for intent, phrases in intent_templates.items():
    encoded_templates[intent] = bert_model.encode(phrases, convert_to_tensor=True)

print(f" Encoded {len(encoded_templates)} intent categories.")
print("Intent categories:", list(intent_templates.keys()))


 Encoded 22 intent categories.
Intent categories: ['soil_low_nitrogen', 'soil_medium_nitrogen', 'soil_high_nitrogen', 'temp_very_cold', 'temp_cool', 'temp_warm', 'temp_hot', 'temp_very_hot', 'humidity_very_low', 'humidity_low', 'humidity_moderate', 'humidity_high', 'humidity_very_high', 'rainfall_very_low', 'rainfall_low', 'rainfall_moderate', 'rainfall_high', 'rainfall_very_high', 'ph_acidic', 'ph_slightly_acidic', 'ph_neutral', 'ph_alkaline']


In [5]:
def extract_features_bert(query, threshold=0.30):
    """
    Extract soil/temperature/humidity/rainfall/pH intent from
    a farmer's natural language query using BERT cosine similarity.

    Args:
        query (str)     : Farmer's natural language input
        threshold (float): Minimum cosine similarity to accept a match (0–1)

    Returns:
        extracted (dict): Best matching intent per feature category
        scores (dict)   : Raw similarity scores for all intents
    """
    query_embedding = bert_model.encode(query, convert_to_tensor=True)

    
    scores = {}
    for intent, template_embeddings in encoded_templates.items():
        sims = util.cos_sim(query_embedding, template_embeddings)
        scores[intent] = sims.max().item()

    
    categories = {
        'nitrogen':    ['soil_low_nitrogen', 'soil_medium_nitrogen', 'soil_high_nitrogen'],
        'temperature': ['temp_very_cold', 'temp_cool', 'temp_warm', 'temp_hot', 'temp_very_hot'],
        'humidity':    ['humidity_very_low', 'humidity_low', 'humidity_moderate',
                        'humidity_high', 'humidity_very_high'],
        'rainfall':    ['rainfall_very_low', 'rainfall_low', 'rainfall_moderate',
                        'rainfall_high', 'rainfall_very_high'],
        'ph':          ['ph_acidic', 'ph_slightly_acidic', 'ph_neutral', 'ph_alkaline'],
    }

    extracted = {}
    for category, intents in categories.items():
        best_intent = max(intents, key=lambda i: scores[i])
        extracted[category] = best_intent if scores[best_intent] >= threshold else None

    return extracted, scores



test_query = "My field barely gets any rain and the soil is very dry with low nutrients"
features, scores = extract_features_bert(test_query)

print("Query   :", test_query)
print("Extracted:", features)
print()
print("Top similarity scores:")
for intent, score in sorted(scores.items(), key=lambda x: -x[1])[:8]:
    print(f"  {intent:<30} {score:.3f}")


Query   : My field barely gets any rain and the soil is very dry with low nutrients
Extracted: {'nitrogen': 'soil_low_nitrogen', 'temperature': 'temp_cool', 'humidity': 'humidity_very_low', 'rainfall': 'rainfall_very_low', 'ph': 'ph_slightly_acidic'}

Top similarity scores:
  rainfall_very_low              0.652
  rainfall_low                   0.646
  humidity_very_low              0.584
  rainfall_high                  0.561
  rainfall_moderate              0.551
  soil_low_nitrogen              0.528
  humidity_low                   0.504
  soil_medium_nitrogen           0.476


In [6]:


NITROGEN_MAP = {
    'soil_low_nitrogen':    20,   # apple, grapes, lentil, orange ~18–21
    'soil_medium_nitrogen': 50,   # lentil, papaya, pigeonpeas ~49–50
    'soil_high_nitrogen':   100,  # banana, coffee, cotton, muskmelon ~99–118
}

TEMPERATURE_MAP = {
    'temp_very_cold': 12.0,   # below dataset Q1 ~18°C
    'temp_cool':      20.0,   # chickpea mean 18.9, kidneybeans 20.1
    'temp_warm':      25.5,   # dataset mean 25.6
    'temp_hot':       31.0,   # mango mean 31.2, papaya 33.7
    'temp_very_hot':  38.0,   # near dataset max 43.7
}

HUMIDITY_MAP = {
    'humidity_very_low': 18.0,   # chickpea mean 16.9, kidneybeans 21.6
    'humidity_low':      52.0,   # mango 50.2, mothbeans 53.2
    'humidity_moderate': 65.0,   # blackgram 65.1, maize 65.1
    'humidity_high':     82.0,   # rice 82.3, cotton 79.8
    'humidity_very_high': 93.0,  # apple 92.3, coconut 94.8, papaya 92.4
}

RAINFALL_MAP = {
    'rainfall_very_low':  28.0,   # muskmelon mean 24.7 — lowest crop
    'rainfall_low':       52.0,   # lentil 45.7, mothbeans 51.2, mungbean 48.4
    'rainfall_moderate':  94.0,   # dataset median ~94.9
    'rainfall_high':      160.0,  # coconut 175.7, coffee 158.1, jute 174.8
    'rainfall_very_high': 240.0,  # rice mean 236.2 — highest crop
}

PH_MAP = {
    'ph_acidic':          5.0,   # apple 5.93, kidneybeans 5.75, pigeonpeas 5.79
    'ph_slightly_acidic': 6.0,   # banana 5.98, coconut 5.98
    'ph_neutral':         6.5,   # dataset mean 6.47
    'ph_alkaline':        7.5,   # blackgram 7.13, chickpea 7.34
}


PK_MAP = {
    'soil_low_nitrogen':    {'P': 67,  'K': 40},   # lentil, kidneybeans avg
    'soil_medium_nitrogen': {'P': 53,  'K': 35},   # dataset mean
    'soil_high_nitrogen':   {'P': 45,  'K': 40},   # banana, cotton avg
}


def generate_feature_vector(extracted):
    """
    Build a [N, P, K, temperature, humidity, ph, rainfall] vector
    from BERT-extracted intent labels, using dataset-calibrated values.
    """

    N           = 50.6
    P           = 53.4
    K           = 48.1
    temperature = 25.6
    humidity    = 71.5
    ph          = 6.47
    rainfall    = 103.5


    if extracted.get('nitrogen') and extracted['nitrogen'] in NITROGEN_MAP:
        N = NITROGEN_MAP[extracted['nitrogen']]
        P = PK_MAP[extracted['nitrogen']]['P']
        K = PK_MAP[extracted['nitrogen']]['K']

    if extracted.get('temperature') and extracted['temperature'] in TEMPERATURE_MAP:
        temperature = TEMPERATURE_MAP[extracted['temperature']]

    if extracted.get('humidity') and extracted['humidity'] in HUMIDITY_MAP:
        humidity = HUMIDITY_MAP[extracted['humidity']]

    if extracted.get('rainfall') and extracted['rainfall'] in RAINFALL_MAP:
        rainfall = RAINFALL_MAP[extracted['rainfall']]

    if extracted.get('ph') and extracted['ph'] in PH_MAP:
        ph = PH_MAP[extracted['ph']]

    return [[N, P, K, temperature, humidity, ph, rainfall]]


sample_extracted = {
    'nitrogen':    'soil_low_nitrogen',
    'temperature': 'temp_warm',
    'humidity':    'humidity_very_high',
    'rainfall':    'rainfall_very_high',
    'ph':          'ph_slightly_acidic',
}
vec = generate_feature_vector(sample_extracted)
cols = ['N', 'P', 'K', 'temperature', 'humidity', 'ph', 'rainfall']
print("Sample feature vector:")
for col, val in zip(cols, vec[0]):
    print(f"  {col:<14}: {val}")


Sample feature vector:
  N             : 20
  P             : 67
  K             : 40
  temperature   : 25.5
  humidity      : 93.0
  ph            : 6.0
  rainfall      : 240.0


In [ ]:
print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=encoder.classes_))


Classification Report:
              precision    recall  f1-score   support

       apple       1.00      1.00      1.00        20
      banana       1.00      1.00      1.00        20
   blackgram       1.00      0.95      0.97        20
    chickpea       1.00      1.00      1.00        20
     coconut       1.00      1.00      1.00        20
      coffee       1.00      1.00      1.00        20
      cotton       1.00      1.00      1.00        20
      grapes       1.00      1.00      1.00        20
        jute       0.95      1.00      0.98        20
 kidneybeans       1.00      1.00      1.00        20
      lentil       1.00      1.00      1.00        20
       maize       0.95      1.00      0.98        20
       mango       1.00      1.00      1.00        20
   mothbeans       1.00      1.00      1.00        20
    mungbean       1.00      1.00      1.00        20
   muskmelon       1.00      1.00      1.00        20
      orange       1.00      1.00      1.00        20
    

## Step 10 — Full Deep Learning NLP Pipeline

Combines:  
1. **BERT** encodes the farmer's query → semantic vector  
2. **Cosine similarity** extracts feature intents  
3. **Lookup tables** (dataset-calibrated) → numeric feature vector  
4. **Random Forest** predicts top-N crops with confidence %  


In [ ]:
def recommend_crops_bert(query, top_n=3, threshold=0.30, verbose=True):
    """
    Full BERT NLP → feature extraction → RF prediction pipeline.

    Args:
        query (str)      : Farmer's natural language question
        top_n (int)      : Number of crop recommendations to return
        threshold (float): BERT cosine similarity threshold
        verbose (bool)   : Print detailed output

    Returns:
        recommendations (list of dict): top_n crops with confidence
        extracted (dict)              : detected feature intents
        explanations (list of str)    : human-readable reasoning
    """
    if verbose:
        print(f" Query: '{query}'")
        print("─" * 65)

    # ── Step 1: BERT NLP extraction ───────────────────────────────────
    extracted, scores = extract_features_bert(query, threshold=threshold)

    if verbose:
        print(" BERT-extracted intents:")
        for cat, intent in extracted.items():
            score = scores.get(intent, 0) if intent else 0
            status = f"{intent}  (sim={score:.3f})" if intent else "❌ not detected"
            print(f"   {cat:<14}: {status}")

    # ── Step 2: Build numeric feature vector ──────────────────────────
    feature_vector = generate_feature_vector(extracted)

    if verbose:
        cols = ['N','P','K','temp','humidity','ph','rainfall']
        vec  = feature_vector[0]
        print(f"\n Feature vector: {dict(zip(cols, [round(v,1) for v in vec]))}")

    # ── Step 3: RF prediction ─────────────────────────────────────────
    probabilities = rf_model.predict_proba(feature_vector)[0]
    top_indices   = np.argsort(probabilities)[-top_n:][::-1]

    recommendations = []
    for idx in top_indices:
        crop       = encoder.inverse_transform([idx])[0]
        confidence = round(probabilities[idx] * 100, 2)
        recommendations.append({'crop': crop, 'confidence': confidence})

    # ── Step 4: Generate explanations ────────────────────────────────
    label_descriptions = {
        'soil_low_nitrogen':    'low-nitrogen soil',
        'soil_medium_nitrogen': 'medium-nitrogen soil',
        'soil_high_nitrogen':   'high-nitrogen fertile soil',
        'temp_very_cold': 'very cold climate (<15°C)',
        'temp_cool':      'cool climate (~20°C)',
        'temp_warm':      'warm climate (~25°C)',
        'temp_hot':       'hot climate (~31°C)',
        'temp_very_hot':  'very hot climate (>35°C)',
        'humidity_very_low':  'very low humidity (<20%)',
        'humidity_low':       'low humidity (~52%)',
        'humidity_moderate':  'moderate humidity (~65%)',
        'humidity_high':      'high humidity (~82%)',
        'humidity_very_high': 'very high humidity (~93%)',
        'rainfall_very_low':  'very low rainfall (~28mm)',
        'rainfall_low':       'low rainfall (~52mm)',
        'rainfall_moderate':  'moderate rainfall (~94mm)',
        'rainfall_high':      'high rainfall (~160mm)',
        'rainfall_very_high': 'very high rainfall (~240mm)',
        'ph_acidic':          'acidic soil (pH ~5.0)',
        'ph_slightly_acidic': 'slightly acidic soil (pH ~6.0)',
        'ph_neutral':         'neutral soil (pH ~6.5)',
        'ph_alkaline':        'alkaline soil (pH ~7.5)',
    }

    explanations = []
    for cat, intent in extracted.items():
        if intent and intent in label_descriptions:
            explanations.append(f"Detected {cat}: {label_descriptions[intent]}")
    if not explanations:
        explanations.append("Using dataset average conditions (no specific features detected)")

    return recommendations, extracted, explanations


def display_results(recommendations, explanations):
    """Print formatted crop recommendations and reasoning."""
    print(f"\n Top {len(recommendations)} Recommended Crops:\n")
    for i, rec in enumerate(recommendations, 1):
        bar = '█' * int(rec['confidence'] / 5)
        print(f"  {i}. {rec['crop']:<16} {rec['confidence']:>6.2f}%  {bar}")
    print("\n Reasoning:")
    for reason in explanations:
        print(f"  • {reason}")
    print()


## Step 11 — Test with Real Farmer Queries

These examples show how BERT understands **paraphrases and synonyms**  
— queries that don't use exact feature names still get correctly interpreted.


In [ ]:
test_queries = [
    # Expected → rice (high rainfall, high humidity, warm)
    "My field gets very heavy monsoon rainfall and the air is always moist and warm",

    # Expected → chickpea / lentil (cool, very low humidity, low rainfall)
    "The weather here is quite cool and the air is very dry with little rain",

    # Expected → coconut / papaya (very high humidity, high rainfall, low N)
    "We have extremely humid conditions near the coast with heavy rains throughout the year",

    # Expected → muskmelon / watermelon (hot, very low rainfall, high N)
    "Very hot climate with almost no rainfall, fertile high-nitrogen soil",

    # Expected → coffee (high rainfall, warm, moderate N)
    "Warm temperature with lots of rain and moderately fertile soil",

    # Expected → apple / grapes (cool, high humidity, acidic soil)
    "Cool weather, soil that is slightly acidic, and good humidity levels",

    # Expected → cotton (hot, medium humidity, high N)
    "The land is very fertile with high nitrogen and the climate is warm to hot",
]

for query in test_queries:
    recs, extracted, explanations = recommend_crops_bert(query, verbose=True)
    display_results(recs, explanations)
    print("=" * 65)


 Query: 'My field gets very heavy monsoon rainfall and the air is always moist and warm'
─────────────────────────────────────────────────────────────────
 BERT-extracted intents:
   nitrogen      : ❌ not detected
   temperature   : temp_hot  (sim=0.506)
   humidity      : humidity_low  (sim=0.574)
   rainfall      : rainfall_very_high  (sim=0.673)
   ph            : ❌ not detected

 Feature vector: {'N': 50.6, 'P': 53.4, 'K': 48.1, 'temp': 31.0, 'humidity': 52.0, 'ph': 6.5, 'rainfall': 240.0}

 Top 3 Recommended Crops:

  1. papaya            25.50%  █████
  2. mango             23.50%  ████
  3. pigeonpeas        21.00%  ████

 Reasoning:
  • Detected temperature: hot climate (~31°C)
  • Detected humidity: low humidity (~52%)
  • Detected rainfall: very high rainfall (~240mm)

 Query: 'The weather here is quite cool and the air is very dry with little rain'
─────────────────────────────────────────────────────────────────
 BERT-extracted intents:
   nitrogen      : ❌ not detected
   

## 💬 Step 12 — Interactive Farmer Chatbot

In [ ]:
print("Crop Recommendation Chatbot")
print("Type your farming conditions in natural language.")
print("Examples: 'hot climate with little rain', 'cool humid soil with heavy rainfall'")
print()

query = input("Farmer Query: ")
recs, extracted, explanations = recommend_crops_bert(query, verbose=True)
display_results(recs, explanations)


🌾 Crop Recommendation Chatbot
Type your farming conditions in natural language.
Examples: 'hot climate with little rain', 'cool humid soil with heavy rainfall'

 Query: 'dry soil and little rain'
─────────────────────────────────────────────────────────────────
 BERT-extracted intents:
   nitrogen      : soil_low_nitrogen  (sim=0.527)
   temperature   : temp_very_hot  (sim=0.335)
   humidity      : humidity_very_low  (sim=0.676)
   rainfall      : rainfall_very_low  (sim=0.668)
   ph            : ph_slightly_acidic  (sim=0.448)

 Feature vector: {'N': 20, 'P': 67, 'K': 40, 'temp': 38.0, 'humidity': 18.0, 'ph': 6.0, 'rainfall': 28.0}

 Top 3 Recommended Crops:

  1. muskmelon         28.50%  █████
  2. kidneybeans       26.50%  █████
  3. mothbeans         13.00%  ██

 Reasoning:
  • Detected nitrogen: low-nitrogen soil
  • Detected temperature: very hot climate (>35°C)
  • Detected humidity: very low humidity (<20%)
  • Detected rainfall: very low rainfall (~28mm)
  • Detected ph: slig

## Step 13 — Visualise BERT Semantic Space (PCA)

Plot how query embeddings cluster by meaning — proving BERT understands  
semantics, not just surface keywords.


In [ ]:
semantic_queries = [
    # (query_text, group_label)
    ("barely any rain drought conditions",          "Low Rainfall"),
    ("very little precipitation dry season",        "Low Rainfall"),
    ("almost no rainfall scarce water",             "Low Rainfall"),
    ("heavy monsoon flooding lots of rain",         "High Rainfall"),
    ("very high precipitation rainy season",        "High Rainfall"),
    ("abundant rainfall tropical downpours",        "High Rainfall"),
    ("very cold freezing frost conditions",         "Cold Climate"),
    ("low temperatures cool temperate weather",     "Cold Climate"),
    ("scorching hot above 35 degrees arid",         "Hot Climate"),
    ("extreme heat very high temperature desert",   "Hot Climate"),
    ("high nitrogen rich fertile dark soil",        "High Nitrogen"),
    ("nitrogen rich loamy earth very productive",   "High Nitrogen"),
    ("low nitrogen sandy poor infertile ground",    "Low Nitrogen"),
    ("nutrient deficient light sandy soil",         "Low Nitrogen"),
    ("very high humidity near 100 percent moist",   "High Humidity"),
    ("extremely humid tropical saturated air",      "High Humidity"),
]

texts  = [q[0] for q in semantic_queries]
labels = [q[1] for q in semantic_queries]

label_colors = {
    'Low Rainfall':  '#FF9800',
    'High Rainfall': '#2196F3',
    'Cold Climate':  '#9C27B0',
    'Hot Climate':   '#F44336',
    'High Nitrogen': '#4CAF50',
    'Low Nitrogen':  '#795548',
    'High Humidity': '#00BCD4',
}

embeddings = bert_model.encode(texts)
pca        = PCA(n_components=2)
reduced    = pca.fit_transform(embeddings)

plt.figure(figsize=(13, 8))
plotted_labels = set()
for i, (x, y) in enumerate(reduced):
    label = labels[i]
    color = label_colors[label]
    lbl   = label if label not in plotted_labels else None
    plt.scatter(x, y, c=color, s=120, zorder=3, label=lbl, edgecolors='white', linewidths=0.8)
    plotted_labels.add(label)
    plt.annotate(texts[i][:40], (x, y), fontsize=7,
                 textcoords="offset points", xytext=(6, 4), color='#333')

plt.legend(fontsize=10, loc='best', framealpha=0.9)
plt.title("BERT Semantic Space — Similar Queries Cluster Together\n(PCA of 384-dim embeddings)",
          fontsize=13, fontweight='bold')
plt.xlabel("PCA Component 1")
plt.ylabel("PCA Component 2")
plt.tight_layout()
plt.savefig('bert_semantic_space.png', dpi=120, bbox_inches='tight')
plt.show()
print("✅ Semantic space plot saved as bert_semantic_space.png")
